# Deploying the Gurobi Avocado Price Optimization Workflow in Nextmv

Gurobi has published a notebook on Avocado price and supply optimization [here](https://colab.research.google.com/github/Gurobi/modeling-examples/blob/master/price_optimization/price_optimization.ipynb). They demonstrate how to use an Ordinary Least Squares (OLS) linear regression model to establish the relationship between price and demand based on data from the Hass Avocado Board. They use this fitted OLS model as input to a price and supply optimization model to optimize the supply and price of avocados by region.

At Nextmv, we have taken their notebook, and adapted it to "Nextmv-ify" their solution to this problem. By integrating with Nextmv, we can deploy and operate models on the platform. Following, we unlock automation, collaboration, scalability, and streamlined decision workflows.

While we're demonstrating this process from a Jupyter notebook for ease of exploration, it's important to note that notebooks are best suited for prototyping rather than production use. In a real-world production setting, you would typically develop and refine your model in a notebook before transitioning to a structured Python project within a managed repository for better version control, collaboration, and operational stability.

Now, let's dive in!

## Getting Started: Connect this notebook to your Nextmv Account 🐰

In order to run this notebook end to end, you will need to create a secret called `NEXTMV_API_KEY` in this colab notebook. You can do this by running through the following steps:

* Visit https://cloud.nextmv.io
* Navigate to `Settings` in top nav bar
* Navigate to `API Keys` in the left nav bar
* Copy your API key
* Return to this Colab notebook
* In the left nav, click on the key icon to expand the Secrets menu
* Click ` + Add new secret`
* Type in `NEXTMV_API_KEY` in to “Name” field and copy your API key into “Value”.
* Then click the “Notebook Access” toggle. It should turn blue when the connection is active



## Part I: Deploy the ML Regressor Model to Nextmv 🐰

First, we need to install some dependencies, like the Nextmv Python SDK.

In [ ]:
%pip install --upgrade "nextmv[all]"
%pip install --upgrade "nextmv-gurobipy"
%pip install statsmodels
%pip install gurobipy
%pip install pandas

We import the required packages.

In [ ]:
import nextmv
from nextmv import cloud
import nextmv_gurobipy as ngp
import json
import os
import statsmodels.api as sm
import statsmodels.formula.api as smf
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from google.colab import userdata
import pandas as pd
import gurobipy as gp
import uuid
from gurobipy import GRB

Let's copy over the data processing code Gurobi included in their original notebook.

In [ ]:
avocado = pd.read_csv('https://raw.githubusercontent.com/Gurobi/modeling-examples/master/price_optimization/HABdata_2019_2022.csv') # dataset downloaded directly from HAB
avocado_old = pd.read_csv('https://raw.githubusercontent.com/Gurobi/modeling-examples/master/price_optimization/kaggledata_till2018.csv') # dataset downloaded from Kaggle
avocado = pd.concat([avocado, avocado_old], ignore_index=True)

# Add the index for each year from 2015 through 2022
avocado['date'] = pd.to_datetime(avocado['date'])
avocado['year'] = pd.DatetimeIndex(avocado['date']).year
avocado['year_index'] = avocado['year'] - 2015
avocado = avocado.sort_values(by='date')

# Define the peak season
avocado['month'] = pd.DatetimeIndex(avocado['date']).month
peak_months = range(2,8)
def peak_season(row):
    return 1 if int(row['month']) in peak_months else 0

avocado['peak'] = avocado.apply(lambda row: peak_season(row), axis=1)

# Scale the number of avocados to millions
avocado['units_sold'] = avocado['units_sold']/1000000

# Select only conventional avocados
avocado = avocado[avocado['type'] == 'Conventional']

avocado = avocado[['date','units_sold','price','region','year','month','year_index','peak']].reset_index(drop = True)
regions = ['Great_Lakes','Midsouth','Northeast','Northern_New_England','SouthCentral','Southeast','West','Plains']
df = avocado[avocado.region.isin(regions)]

for col in df.select_dtypes(include=['datetime']).columns:
    df[col] = df[col].astype(str)

Here is where we make very minor modifications to support deployment on the Nextmv Platform.

- We follow the Nextmv convention and define a decision model using the `nextmv.Model` class.
- We define a `solve` function on this class which conforms to the expected signature: reading in a `nextmv.Input` and writing a `nextmv.Output`.

Following this convention unlocks the ability to push up a model object directly from your notebook environment.

Define the `MLRegressorModel` class. All we're doing here is copying code from above and "Nextmv-ifying" it.

In [ ]:
# >>>>>>>>>>>> Start Nextmv-ifying
class MLRegressorModel(nextmv.Model):
    def solve(self, input: nextmv.Input) -> nextmv.Output:
        nextmv.redirect_stdout()
        df = pd.DataFrame(input.data)
# <<<<<<<<<<<<< Stop Nextmv-ifying
        train, test = train_test_split(df, train_size=0.8, random_state=1)
        df_train = pd.DataFrame(train, columns=df.columns)
        df_test = pd.DataFrame(test, columns=df.columns)

        # Train the model
        formula = 'units_sold ~ price + year_index + C(region)+ peak'
        mod = smf.ols(formula,data=df_train)
        result = mod.fit()
        result.summary()

        # Get R^2 from test data
        y_true = df_test['units_sold']
        y_pred = result.predict(df_test)

        formula = 'units_sold ~ price + year_index + C(region)+ peak'
        mod_full = smf.ols(formula,data=df)
        result_full = mod_full.fit()

        y_true_full = df['units_sold']
        y_pred_full = result_full.predict(df)

        # Get the weights and store it
        coef_dict = result_full.params.to_dict()
        coef_dict['C(region)[T.Great_Lakes]'] = 0

        # >>>>>>>>>>>> Start Nextmv-ifying
        statistics = nextmv.Statistics(
            result=nextmv.ResultStatistics(
                custom={
                    "r2_test": r2_score(y_true, y_pred),
                    "r2_full": r2_score(y_true_full, y_pred_full),
                },
            ),
        )

        return nextmv.Output(
            options=nextmv.Options(),
            solution=coef_dict,
            statistics=statistics,
        )
      # <<<<<<<<<<<<< Stop Nextmv-ifying


We can run this model locally, before deploying it to the Nextmv Platform. Since our model takes in a `nextmv.Input`, we can create it by passing in our `df` and some default `nextmv.Options`.

In [ ]:
model = MLRegressorModel()
input = nextmv.Input(data=df.to_dict(), options=nextmv.Options())
output = model.solve(input)
nextmv.write_local(output)

Now that we've confirmed everything works locally, we're ready to deploy to the Nextmv Platform. We do this by creating a Nextmv `cloud.Client` and configuring it with the Nextmv API key (step mentioned in the prerequisites).




In [ ]:
client = cloud.Client(api_key=userdata.get('NEXTMV_API_KEY'))

Once we create the client, we can:
* Specify the app we want to push up to. In this case, we're pushing up to the `avocado-ml-regressor` app.
* Define the model configuration.
* Create an app manifest based on that configuration.
* Push the local model up to the `avocado-ml-regressor` app in Nextmv Cloud. This step may take a few minutes depending on your network speed.

In [ ]:
reg_app_name = "avocado-ml-regressor"
if cloud.Application.exists(client, id=reg_app_name):
    regressor_app = cloud.Application(client=client, id=reg_app_name)
else:
    regressor_app = cloud.Application.new(client=client, id=reg_app_name, name=reg_app_name)

model_configuration = nextmv.ModelConfiguration(
    name=reg_app_name,
    requirements=[
        "nextmv==0.25.0",
        "statsmodels==0.14.4",
        "scikit-learn==1.6.1",
        "pandas==2.2.2"
    ],
    options=None,
)
manifest = cloud.Manifest.from_model_configuration(model_configuration)

regressor_app.push(
    manifest=manifest,
    model=model,
    model_configuration=model_configuration,
    verbose=True,
)

The app is deployed, let’s create a remote run on the Nextmv Platform and print the results.

In [ ]:
regressor_result = regressor_app.new_run_with_result(input=df.to_dict())
print(json.dumps(regressor_result.output, indent=2))

And that's it 🚀! We've got our fitted regression model running as a standalone application on the Nextmv Platform.

## Part II: Deploy the Price and Supply Optimization Model to Nextmv  🐰

Let's try this again with the optimization model. Just as with the ML regressor, we'll:

* Copy the notebook model code into a class, which inherits from `nextmv.Model`.
* Make minor modifications to "Nextmv-ify" it.

Additionally, we are going to:
* Expose an option called `supply`, so we can set the total amount of avocado supply external to the model.
* Define a custom visualization on the app.

  Nextmv works with `Chart.js`, `Plotly`, and `GeoJSON` visualization assets. We've refactored the original Gurobi visualization of the solution to work with `Plotly`, and we've returned this visualization as a custom asset on the run. Custom assets make it possible for you to manage your visualization alongside your model code!

In [ ]:
# >>>>>>>>>>>> Start Nextmv-ifying
class AvocadoPriceDecisionModel(nextmv.Model):
    def solve(self, input: nextmv.Input) -> nextmv.Output:
        data = input.data
        B = input.options.supply # total amount of avocado supply

        m = ngp.Model(input.options)
# <<<<<<<<<<<<< Stop Nextmv-ifying

        # Sets and parameters
        R = data["regions"]   # set of all regions

        peak_or_not = data["peak"] # 1 if it is the peak season; 1 if isn't
        year = data["year"]

        c_waste = data["cost_per_wasted_product"] # the cost ($) of wasting an avocado
        c_transport = data["transport_costs"] # the cost of transporting an avocado

        # Get the lower and upper bounds from the dataset for the price and the number of products to be stocked
        a_min = data["minimum_product_price"] # minimum avocado price in each region
        a_max = data["maximum_product_price"] # maximum avocado price in each region
        b_min = data["minimum_product_allocations"]  # minimum number of avocados allocated to each region
        b_max = data["maximum_product_allocations"]   # maximum number of avocados allocated to each region

        p = m.addVars(R,name="p",lb=a_min, ub=a_max)   # price of avocados in each region
        x = m.addVars(R,name="x",lb=b_min,ub=b_max)  # quantity supplied to each region
        s = m.addVars(R,name="s",lb=0)   # predicted amount of sales in each region for the given price
        w = m.addVars(R,name="w",lb=0)   # excess wasteage in each region

        d = {r: (data["coefficients"]['Intercept']+data["coefficients"]['price']*p[r] + data["coefficients"]['C(region)[T.%s]'%r] + data["coefficients"]['year_index']*(year-2015) + data["coefficients"]['peak']*peak_or_not) for r in R}

        m.setObjective(sum(p[r]*s[r] - c_waste*w[r] - c_transport[r]*x[r] for r in R))
        m.ModelSense = GRB.MAXIMIZE

        m.addConstr(sum(x[r] for r in R) == B)
        m.addConstrs((s[r] <= x[r] for r in R))
        m.addConstrs((s[r] <= d[r] for r in R))
        m.addConstrs((w[r] == x[r]-s[r] for r in R))
        m.Params.NonConvex = 2
        m.optimize()

        solution = pd.DataFrame()
        solution['Region'] = R
        solution['Price'] = [p[r].X for r in R]
        solution['Allocated'] = [round(x[r].X,8) for r in R]
        solution['Sold'] = [round(s[r].X,8) for r in R]
        solution['Wasted'] = [round(w[r].X,8) for r in R]
        solution['Pred_demand'] = [(data["coefficients"]['Intercept']+data["coefficients"]['price']*p[r].X + data["coefficients"]['C(region)[T.%s]'%r] + data["coefficients"]['year_index']*(year-2015) + data["coefficients"]['peak']*peak_or_not) for r in R]

        fig = px.scatter(
            solution,
            x="Price",
            y="Sold",
            color="Region",
            size="Sold",  # Size based on sold quantity
            size_max=15,  # Adjust for desired size of markers
            title="Avocado Sales and Waste by Region",
            labels={"Price": "Price per avocado ($)", "Sold": "Number of avocados sold (millions)"},
        )

        colors = px.colors.qualitative.Plotly  # Use a color palette from Plotly
        num_regions = len(solution["Region"].unique())
        region_colors = {region: colors[i % len(colors)] for i, region in enumerate(solution["Region"].unique())}


        fig.add_trace(
            go.Scatter(
                x=solution["Price"],
                y=solution["Wasted"],
                mode="markers",
                marker=dict(symbol="x", size=10, color=[region_colors[region] for region in solution["Region"]]),  # Assign colors based on region
                name="Wasted",
                showlegend=False,  # Hide legend for wasted points
            )
        )

        fig.update_layout(
            yaxis_range=[0, 5],
            xaxis_range=[1, 2.2],
            legend=dict(x=1.25, y=0.5),  # Adjust legend position
        )

        json_plot = fig.to_json()

# >>>>>>>>>>>> Start Nextmv-ifying
        statistics = ngp.ModelStatistics(m)
        statistics.result.custom = {
            "variables": m.NumVars,
            "constraints": m.NumConstrs,
            "total_waste": sum(w[r].X for r in R),
        }

        asset = nextmv.Asset(
            name="Pricing Charts",
            content_type="json",
            visual=nextmv.Visual(
                visual_schema=nextmv.VisualSchema(value="plotly"),
                label="Pricing Charts",
                visual_type="custom-tab",
            ),
            content=[json.loads(json_plot)],
        )

        return nextmv.Output(
            options=input.options,
            solution=solution.to_dict(),
            statistics=statistics,
            assets=[asset]
        )
# <<<<<<<<<<<<< Stop Nextmv-ifying


Let's run this optimization model locally, as before. You can define the data to use when making the run. Again, we pulled this data from the original Gurobi notebook and just stored it in a `dict` here. Note how we use the result of the ML regressor to populate this data `dict`.

In [ ]:
data = {
    "regions": [
        "Great_Lakes",
        "Midsouth",
        "Northeast",
        "Northern_New_England",
        "SouthCentral",
        "Southeast",
        "West",
        "Plains"
    ],
    "total_amount_of_supply": 30,
    "cost_per_wasted_product": 0.1,
    "peak": 1,
    "transport_costs": {
        'Great_Lakes': .3,
        'Midsouth':.1,
        'Northeast':.4,
        'Northern_New_England':.5,
        'SouthCentral':.3,
        'Southeast':.2,
        'West':.2,
        'Plains':.2
    },
    "year": 2022,
    "minimum_product_price": 0,
    "maximum_product_price": 2,
    "minimum_product_allocations": dict(df.groupby('region')['units_sold'].min()),
    "maximum_product_allocations": dict(df.groupby('region')['units_sold'].max()),
    "coefficients": regressor_result.output.get('solution'),
}


Let's define our options. In this case, the model expects an option called `supply` which specifies the total amount of avocado supply. We are merging all the supported Gurobi options with this custom option to create the final set of options for our app.

In [ ]:
gp_opt = ngp.ModelOptions().to_nextmv()
nm_opt = nextmv.Options(
    nextmv.Option(name="supply", option_type=int, default=30, description="Total amount of avocado supply.", required=False),
)
options = nm_opt.merge(gp_opt)

To run locally, we instantiate the `nextmv.Input`, the decision model, and solve.

In [ ]:
input = nextmv.Input(data=data, options=options)
model = AvocadoPriceDecisionModel()
output = model.solve(input)
nextmv.write_local(output)

All went well locally, now let's push this second model object up to Nextmv Cloud. As before, we need to specify a model configuration and app manifest before pushing. This time, we additionally demonstrate how to cut a version and assign that version to a managed instance called `staging`.

In [ ]:
optimizer_app_name = "avocado-price-optimizer"
if cloud.Application.exists(client, id=optimizer_app_name):
    optimizer_app = cloud.Application(client=client, id=optimizer_app_name)
else:
    optimizer_app = cloud.Application.new(client=client, id=optimizer_app_name, name=optimizer_app_name)

model_configuration = nextmv.ModelConfiguration(
    name="avocado-price-optimizer",
    requirements=[
        "nextmv==0.25.0",
        "nextmv-gurobipy==0.2.1",
        "plotly==6.0.0"
    ],
    options=options,
)
manifest = nextmv.cloud.Manifest.from_model_configuration(model_configuration)

optimizer_app.push(
    manifest=manifest,
    model=model,
    model_configuration=model_configuration,
    verbose=True,
)

version = str(uuid.uuid4())
optimizer_app.new_version(id=version, name=version)
optimizer_app.new_instance(
    version_id=version,
    id="staging",
    name="staging"
)

Now, let's try running it remotely on the  Nextmv Platform.

In [ ]:
optimization_result = optimizer_app.new_run_with_result(input=data)
print(json.dumps(optimization_result.output, indent=2))

## Part III: Create a Nextmv Workflow to Chain the Apps Together 🐰

Up to this point, we have two distinctly managed Nextmv Applications:
* `avocado-ml-regressor`
* `avocado-price-optimizer`

Now, we will create a third Nextmv Application to run a workflow which chains these executions together:

Fit the regressor ➡️ Send fitted results ➡️ Price optimization

Let's set up that workflow using Nextmv and `nextpipe`!



---



Write the data to the local filesystem, under the `input` folder.

In [ ]:
import pandas as pd
avocado = pd.read_csv('https://raw.githubusercontent.com/Gurobi/modeling-examples/master/price_optimization/HABdata_2019_2022.csv') # dataset downloaded directly from HAB
avocado_old = pd.read_csv('https://raw.githubusercontent.com/Gurobi/modeling-examples/master/price_optimization/kaggledata_till2018.csv') # dataset downloaded from Kaggle
avocado = pd.concat([avocado, avocado_old], ignore_index=True)
if not os.path.exists('input'):
    os.mkdir('input')

avocado.to_csv('input/avocado_input.csv', index=False)

Install `nextpipe`.

In [ ]:
%pip install --upgrade nextpipe

We're ready to construct our workflow. The workflow is intended to run as a Nextmv Application.

These are the steps of the workflow. Note how each step specifies the predecessors, chaining the logic together.

| Step | Needs | Description |
| :-: | :- | :- |
| `load` |  | Load CSV data and transform it into a `pd.DataFrame`. |
| ⬇️ |  |  |
| `prepare` | `load` | Receive the `pd.DataFrame`, do all the data preparation form earlier, an return a `dict`. |
| ⬇️ |  |  |
| `regress` | `prepare` | Call the deployed Nextmv Application to fit the ML regressor model. We do this by leveraging the `@app` decorator from `nextpipe`. |
| ⬇️ |  |  |
| `join` | `prepare`, `regress` | Join the output from the `regress` step with the optimization input data. |
| ⬇️ |  |  |
| `optimize` | `join` | Call the deployed Nextmv Application to optimize the price and supply of avocados per region using the `@app` decorator from `nextpipe`. |
| ⬇️ |  |  |
| `postprocess` | `optimize` | Transform the optimization output to a `nextmv.Output`. |     



---



We are going to create a new `workflow` folder, and store the Nextmv Application files in it:
* `main.py` ➡️ File with the Nextmv Application code.
* `requirements.txt` ➡️ File that specifies the dependencies the app needs.

The previous applications were created from a `nextmv.Model`, but in this case, we are creating an app based on files.

In [ ]:
if not os.path.exists('workflow'):
    os.mkdir('workflow')

Write the `main.py` file to the `workflow` folder.

In [ ]:
%%writefile workflow/main.py
import os

import nextmv
import pandas as pd
from nextmv import cloud
from nextpipe import FlowSpec, app, needs, step

options = nextmv.Options(
    nextmv.Option(
        name="input",
        option_type=str,
        default="input",
        description="Path to the input data.",
        required=False,
    ),
    nextmv.Option(
        name="supply",
        option_type=int,
        default=30,
        description="Total amount of avocado supply.",
        required=False,
    ),
)


class Flow(FlowSpec):
    @step
    def load(_):
        """Loads the data."""
        csv_file = [f for f in os.listdir(options.input) if f.endswith(".csv")][0]
        avocado = pd.read_csv(os.path.join(options.input, csv_file))
        return avocado

    @needs(predecessors=[load])
    @step
    def prepare(avocado: pd.DataFrame):
        """Prepares the data."""
        # Add the index for each year from 2015 through 2022
        avocado["date"] = pd.to_datetime(avocado["date"])
        avocado["year"] = pd.DatetimeIndex(avocado["date"]).year
        avocado["year_index"] = avocado["year"] - 2015
        avocado = avocado.sort_values(by="date")

        # Define the peak season
        avocado["month"] = pd.DatetimeIndex(avocado["date"]).month
        peak_months = range(2, 8)

        def peak_season(row):
            return 1 if int(row["month"]) in peak_months else 0

        avocado["peak"] = avocado.apply(lambda row: peak_season(row), axis=1)

        # Scale the number of avocados to millions
        avocado["units_sold"] = avocado["units_sold"] / 1000000

        # Select only conventional avocados
        avocado = avocado[avocado["type"] == "Conventional"]

        avocado = avocado[
            [
                "date",
                "units_sold",
                "price",
                "region",
                "year",
                "month",
                "year_index",
                "peak",
            ]
        ].reset_index(drop=True)
        regions = [
            "Great_Lakes",
            "Midsouth",
            "Northeast",
            "Northern_New_England",
            "SouthCentral",
            "Southeast",
            "West",
            "Plains",
        ]
        df = avocado[avocado.region.isin(regions)]

        for col in df.select_dtypes(include=["datetime"]).columns:
            df[col] = df[col].astype(str)

        return df.to_dict()

    @app(app_id="avocado-ml-regressor", instance_id="devint")
    @needs(predecessors=[prepare])
    @step
    def regress():
        """Fits the ML regressor model."""
        pass

    @needs(predecessors=[prepare, regress])
    @step
    def join(df: dict, coef_dict: dict):
        """Joins the ML regressor results with the optimization input data."""
        df = pd.DataFrame(df)
        data = {
            "regions": [
                "Great_Lakes",
                "Midsouth",
                "Northeast",
                "Northern_New_England",
                "SouthCentral",
                "Southeast",
                "West",
                "Plains",
            ],
            "total_amount_of_supply": 30,
            "cost_per_wasted_product": 0.1,
            "peak": 1,
            "transport_costs": {
                "Great_Lakes": 0.3,
                "Midsouth": 0.1,
                "Northeast": 0.4,
                "Northern_New_England": 0.5,
                "SouthCentral": 0.3,
                "Southeast": 0.2,
                "West": 0.2,
                "Plains": 0.2,
            },
            "year": 2022,
            "minimum_product_price": 0,
            "maximum_product_price": 2,
            "minimum_product_allocations": dict(
                df.groupby("region")["units_sold"].min()
            ),
            "maximum_product_allocations": dict(
                df.groupby("region")["units_sold"].max()
            ),
            "coefficients": coef_dict.get("solution"),
        }

        return data

    @app(
        app_id="avocado-price-optimizer",
        instance_id="staging",
        parameters={"supply": options.supply},
    )
    @needs(predecessors=[join])
    @step
    def optimize():
        """Optimizes the price and supply of avocados per region."""
        pass

    @needs(predecessors=[optimize])
    @step
    def postprocess(result: dict):
        """Postprocesses the results."""
        tabular = pd.DataFrame(result.get("solution"))
        output = nextmv.Output(
            output_format=nextmv.OutputFormat.CSV_ARCHIVE,
            options=result.get("solution", {}).get("options"),
            solution={"solution": tabular.to_dict(orient="records")},
            statistics=result.get("statistics", {}),
            assets=result.get("assets", []),
        )
        return output


def main() -> None:
    """Main entrypoint for the program."""

    client = cloud.Client(api_key=os.getenv("NEXTMV_API_KEY"))
    flow = Flow(name="DecisionFlow", input=None, conf=None, client=client)
    flow.run()
    output = flow.get_result(flow.postprocess)
    nextmv.write_local(output)

if __name__ == "__main__":
    main()

Write the `requirements.txt` file to the `workflow` folder.

In [ ]:
%%writefile workflow/requirements.txt
pandas==2.2.2
nextmv==0.25.0
nextpipe==0.1.3

Lastly, and similar to the previous applications, we create our `workflow` Nextmv Application. For this application, we are also:

* Creating a secrets collection with the value of our `NEXTMV_API_KEY`
* Cutting a version of our executable
* Creating a managed instance to make runs against
* Configuring the instance with to the newly cut version and secrets collection

In [ ]:
workflow_app_name = "workflow"
if cloud.Application.exists(client, id=workflow_app_name):
    workflow_app = cloud.Application(client=client, id=workflow_app_name)
else:
    workflow_app = cloud.Application.new(client=client, id=workflow_app_name, name=workflow_app_name, is_workflow=True)

# This manifest ends up as an `app.yaml` file in the app.
manifest = cloud.Manifest(
    type=cloud.ManifestType.PYTHON,
    runtime=cloud.ManifestRuntime.PYTHON,
    files=["main.py"],
    python=cloud.ManifestPython(
        pip_requirements="requirements.txt"
    )
)

workflow_app.push(
    manifest=manifest,
    app_dir="workflow",
    verbose=True,
)

secrets = workflow_app.new_secrets_collection(
    secrets=[
        cloud.Secret(
            secret_type=cloud.SecretType.ENV,
            location="NEXTMV_API_KEY",
            value=userdata.get("NEXTMV_API_KEY"),
        ),
    ],
    id="workflow-secrets",
    name="workflow-secrets",
)

version = str(uuid.uuid4())
workflow_app.new_version(id=version, name=version)
workflow_app.new_instance(
    version_id=version,
    id="staging",
    name="staging",
    configuration=cloud.InstanceConfiguration(
        secrets_collection_id="workflow-secrets"
    ),
)

Congrats, you're all done 🚀!

You now have 3 apps deployed in your Nextmv Account, go [here](https://cloud.nextmv.io/) to experiment!